# HumanoidBench 自训通关 · *Self-Trained Showcase*

区别于 `HumanoidBench-Showcase.ipynb`（**baseline** dmux/DR.Q 9 task 公开 ckpt 通关展示），
本 notebook 专注于 **`wsagi/HumanoidBench-DR.Q`** —— 我们从零自训、**超越官方 baseline** 的 ckpt：

_Sibling notebook to `HumanoidBench-Showcase.ipynb` (which showcases the **public** dmux/DR.Q baseline).
This notebook focuses on **`wsagi/HumanoidBench-DR.Q`** — our from-scratch self-trained ckpts that **beat** the official baseline._

| Task | 自训 / Self-trained | dmux baseline | 提升 / Gain |
| --- | --- | --- | --- |
| `h1-walk-v0` | **90% / mean 801** | ~30% / mean ~530 | **3× 成功率** |
| `g1-walk-v0` | **70% / mean 711** | 0% (torque baseline mean ~100) | **7.1× return** |

🤗 HF repo: <https://huggingface.co/wsagi/HumanoidBench-DR.Q>

> **每个代码 cell 自包含**，可以任意顺序执行 / **Every code cell is self-contained — runs in any order.**


## §1  从 HF 下载自训 ckpt / *Download self-trained ckpts from HF*

拉取 `encoder.pt + policy.pt + agent_var.npy`（推理只需 ~13 MB / task），跳过 optimizer/target（76 MB / task，续训才需要）。

_Pulls the 3 inference-critical files (~13 MB/task), skips the 76 MB optimizer/target tail used only for resume-training._


In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

REPO = "wsagi/HumanoidBench-DR.Q"
TASKS = ["h1-walk-v0", "g1-walk-v0"]

snap_root = Path(snapshot_download(
    repo_id=REPO,
    allow_patterns=[
        *[f"DRQ+HBench-{t}+0/encoder.pt"     for t in TASKS],
        *[f"DRQ+HBench-{t}+0/policy.pt"      for t in TASKS],
        *[f"DRQ+HBench-{t}+0/agent_var.npy"  for t in TASKS],
        "eval/*.jsonl",
        "README.md",
    ],
))
print(f"✅ HF snapshot @ {snap_root}")
for f in sorted(snap_root.rglob("*")):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  {f.relative_to(snap_root)}  ({size:.1f} KB)")

## §2  一键 GUI 观看（自训 ckpt） / *One-click GUI playback (self-trained)*

前置：`bash patches/apply.sh` 已对 `dependencies/dr-q` 和 `dependencies/humanoid-bench` 打过补丁（G1 必需）。

本 cell **自动从 HF 拉自训 ckpt**（cache 命中秒返回），用**当前 notebook 内核的 Python**（`sys.executable`）启动 `scripts/drq_viewer.py`，跑的就是 wsagi 自训权重。

_Uses the **notebook kernel's own Python** (`sys.executable`) — no `conda run` to swallow errors. Subprocess stdout/stderr go to a log file under `.viewer_logs/` so you can `tail -f` if something goes wrong._


In [ ]:
import os, sys, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download
from ipywidgets import Button, VBox, Output
from IPython.display import display

REPO = "wsagi/HumanoidBench-DR.Q"
TASKS = ["h1-walk-v0", "g1-walk-v0"]
ROOT = Path.cwd()                                        # notebook lives at repo root
LOG_DIR = ROOT / ".viewer_logs"
LOG_DIR.mkdir(exist_ok=True)

# Self-contained: pull (or cache-hit) inference-critical files for both tasks
snap_root = Path(snapshot_download(
    repo_id=REPO,
    allow_patterns=[
        *[f"DRQ+HBench-{t}+0/encoder.pt"     for t in TASKS],
        *[f"DRQ+HBench-{t}+0/policy.pt"      for t in TASKS],
        *[f"DRQ+HBench-{t}+0/agent_var.npy"  for t in TASKS],
    ],
))

SELF_TRAINED = [
    ("h1-walk-v0", 0, "H1 走路 / Walk (self-trained, 90% success, mean 801)"),
    ("g1-walk-v0", 0, "G1 走路 / Walk (self-trained, 70% success, mean 711)"),
]
out = Output()

def launch(task, seed):
    ckpt_dir = snap_root / f"DRQ+HBench-{task}+{seed}"
    log_path = LOG_DIR / f"{task}-seed{seed}.log"
    def _click(_b):
        # PYTHONUNBUFFERED so log shows progress live; DISPLAY=:0 for local X11 viewer
        env = {**os.environ, "DISPLAY": os.environ.get("DISPLAY", ":0"), "PYTHONUNBUFFERED": "1"}
        log_f = log_path.open("w")
        proc = subprocess.Popen(
            [sys.executable, "scripts/drq_viewer.py",
             "--task", task, "--seed", str(seed),
             "--ckpt_dir", str(ckpt_dir),
             "--action_repeat", "2", "--fps", "60"],
            cwd=str(ROOT),
            env=env,
            stdout=log_f, stderr=subprocess.STDOUT,
        )
        with out:
            print(f"▶ launched task={task} seed={seed}  pid={proc.pid}")
            print(f"   ckpt_dir = {ckpt_dir}")
            print(f"   log      = {log_path}   (tail -f to debug)")
            print(f"   stop     = !kill {proc.pid}")
    return _click

buttons = []
for task, seed, label in SELF_TRAINED:
    b = Button(description=f"▶ {label}", layout={"width": "600px"})
    b.on_click(launch(task, seed))
    buttons.append(b)

display(VBox(buttons), out)

## §3  自训 vs 公开 baseline 数据对照 / *Self-trained vs public baseline*


In [ ]:
import json
from pathlib import Path
from huggingface_hub import snapshot_download

REPO = "wsagi/HumanoidBench-DR.Q"
snap_root = Path(snapshot_download(repo_id=REPO, allow_patterns=["eval/*.jsonl"]))

# 自训成绩单（HF snapshot eval/*.jsonl 最后一行是 _summary） / self-trained summary rows
rows = []
for jl in sorted((snap_root / "eval").glob("*.jsonl")):
    for line in jl.open():
        d = json.loads(line)
        if d.get("_summary"):
            rows.append({
                "task": d["task"],
                "success%": f"{d['success_rate']*100:.0f}%",
                "mean_return": f"{d['mean_return']:.1f}",
                "mean_steps": f"{d['mean_steps']:.0f}",
                "N (eps × seeds)": f"{d['eval_per_seed']} × {len(d['seeds'])}",
                "source": "wsagi self-trained",
            })

# 对照官方 baseline（数字来自 results/drq_multiseed/h1-walk-v0.jsonl + brainstorm 文档记录的 torque G1）
rows.append({"task":"h1-walk-v0","success%":"73%","mean_return":"≈ 530","mean_steps":"—",
             "N (eps × seeds)":"10 × 3","source":"dmux/DR.Q public"})
rows.append({"task":"g1-walk-v0","success%":"0%","mean_return":"≈ 100","mean_steps":"—",
             "N (eps × seeds)":"10 × 1","source":"DR.Q torque (no patch)"})

try:
    import pandas as pd
    df = pd.DataFrame(rows).sort_values(by=["task","source"]).reset_index(drop=True)
    display(df)
except ImportError:
    for r in rows:
        print(r)

## §4  G1 必备 patches / *Required patches for G1*

G1 通关**不是开箱即用**，单纯换 G1 + 默认 DR.Q 配置训练 1M 步 success=0% / mean ~100。
三模型 brainstorm（Opus + GPT-5.5 + DeepSeek-V4-Pro）后定位两层根因：

_G1 is **not** plug-and-play — three rounds of brainstorming (Opus + GPT-5.5 + DeepSeek-V4-Pro) identified two layers of root cause._

| Layer | Patch | What | Why |
| --- | --- | --- | --- |
| 1 | `patches/g1-pos-control.patch` | G1 motor → **PD position control** | torque control 对 sample efficiency 不友好；PD 与 H1 一致 |
| 2 | `patches/humanoid-bench-g1-blocked-hands.patch` | 扩展 `BlockedHandsLocoWrapper` 支持 G1（37D → 23D action） | DR.Q σ=0.2 isotropic noise 在 14 维手指上污染 encoder dynamics loss → 250k 步 catastrophic forgetting |

实证 / Evidence:

| Round | Config | act_dim | 500k 步 result |
| --- | --- | --- | --- |
| 1 | torque baseline | 37 | mean 100, success 0% (**DEAD**) |
| 2 | PD only (Tier S) | 37 | mean 435, success 0% (**4.3× 提升但未通关**) |
| 3 | PD + BlockedHands (Tier S') | **23** | **mean 711, success 70% ✅** |

完整分析见 [`docs/g1_training_strategies.html`](docs/g1_training_strategies.html)（HTML + SVG，单文件可分享）。


## §5  训练复现 / *Reproduce training*

完整流水线 = **训练 + watcher + ckpt eval daemon** 三进程并行，**禁止** 启动后离场（LeIsaac 经验）。

_Full pipeline = train + slice-based watcher + per-ckpt eval daemon — never fire-and-forget._

```bash
git clone --recursive https://github.com/vitorcen/humanoid-training
cd humanoid-training && bash patches/apply.sh

# H1-walk: ~6.6h on RTX 4090, 500k steps, success 90% / mean 801
cd dependencies/dr-q/DRQ && nohup python main.py \
    --env HBench-h1-walk-v0 --seed 0 \
    --total_timesteps 500000 --save_freq 50000 \
    --base_folder $PWD/../../../runs/h1_walk_pilot/ --save_experiment \
    > ../../../runs/h1_walk_pilot/train.log 2>&1 &

# G1-walk: ~3.0h on RTX 4090, 500k steps, success 70% / mean 711
#   (patches apply already enables blocked_hands=True for G1)
cd dependencies/dr-q/DRQ && nohup python main.py \
    --env HBench-g1-walk-v0 --seed 0 \
    --total_timesteps 500000 --save_freq 50000 \
    --base_folder $PWD/../../../runs/g1_walk_pdbh_pilot/ --save_experiment \
    > ../../../runs/g1_walk_pdbh_pilot/train.log 2>&1 &
```

Watcher + daemon 命令见根目录 `README.md` 的「自训流水线」段。
